# 02 — Data Cleaning, Alignment, and Indicator Construction (θ)

This notebook implements:

US3 — Data Cleaning & Alignment  
US4 — Labour Market Imbalance Indicator Construction

The objective is to construct a consistent, transparent, and reproducible
monthly labour market dataset suitable for:

• Descriptive analysis  
• Diagnostic analysis  
• Predictive forecasting  
• Scenario modeling  

---

Compute labour market imbalance indicator:
   
   θₜ = Vₜ / Uₜ

   where:
   - Vₜ = job vacancies (persons)
   - Uₜ = unemployed persons

To ensure measurement consistency:

- Unemployment is converted from thousands to persons.
- Vacancies are already expressed in persons.

---

## COVID Structural Shock Treatment

Following supervisory guidance:

• The April–September 2020 period is retained for descriptive and diagnostic analysis.
• The same period is excluded from predictive modeling to preserve trend stability.

Two datasets will therefore be exported:

1. `final_dataset_full.csv`
2. `final_dataset_modeling.csv`


In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# Load raw datasets
lfs_raw = pd.read_csv("../1_data/raw/labour_force_raw.csv.csv")
vac_raw = pd.read_csv("../1_data/raw/job vacancies.csv")

print("LFS raw shape:", lfs_raw.shape)
print("Vacancy raw shape:", vac_raw.shape)

display(lfs_raw.head(3))
display(vac_raw.head(3))



LFS raw shape: (468, 19)
Vacancy raw shape: (111, 16)


,REF_DATE,GEO,DGUID,Labour force characteristics,Gender,Age group,Statistics,Data type,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,2015-04,Canada,2021A000011124,Population,Total - Gender,15 years and over,Estimate,Seasonally adjusted,Persons in thousands,428,thousands,3,v2062809,1.1.1.1.1.1,29000.4,NaN,NaN,NaN,1
1,2015-05,Canada,2021A000011124,Population,Total - Gender,15 years and over,Estimate,Seasonally adjusted,Persons in thousands,428,thousands,3,v2062809,1.1.1.1.1.1,29020.2,NaN,NaN,NaN,1
2,2015-06,Canada,2021A000011124,Population,Total - Gender,15 years and over,Estimate,Seasonally adjusted,Persons in thousands,428,thousands,3,v2062809,1.1.1.1.1.1,29048.8,NaN,NaN,NaN,1


,REF_DATE,GEO,DGUID,North American Industry Classification System (NAICS),Statistics,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,2015-04,Canada,2016A000011124,"Total, all industries",Job vacancies,Number,223,units,0,v1446283287,1.1.1,440425,A,NaN,NaN,0
1,2015-05,Canada,2016A000011124,"Total, all industries",Job vacancies,Number,223,units,0,v1446283287,1.1.1,407040,A,NaN,NaN,0
2,2015-06,Canada,2016A000011124,"Total, all industries",Job vacancies,Number,223,units,0,v1446283287,1.1.1,398910,A,NaN,NaN,0


In [5]:
lfs_filtered = lfs_raw[
    (lfs_raw["GEO"] == "Canada") &
    (lfs_raw["Gender"] == "Total - Gender") &
    (lfs_raw["Age group"] == "15 years and over") &
    (lfs_raw["Data type"] == "Seasonally adjusted") &
    (lfs_raw["Labour force characteristics"].isin(
        ["Employment", "Unemployment", "Labour force", "Population"]
    ))
].copy()

print("Filtered LFS shape:", lfs_filtered.shape)
print(lfs_filtered["Labour force characteristics"].unique())

display(lfs_filtered.head())


Filtered LFS shape: (468, 19)
['Population' 'Labour force' 'Employment' 'Unemployment']


,REF_DATE,GEO,DGUID,Labour force characteristics,Gender,Age group,Statistics,Data type,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,2015-04,Canada,2021A000011124,Population,Total - Gender,15 years and over,Estimate,Seasonally adjusted,Persons in thousands,428,thousands,3,v2062809,1.1.1.1.1.1,29000.4,NaN,NaN,NaN,1
1,2015-05,Canada,2021A000011124,Population,Total - Gender,15 years and over,Estimate,Seasonally adjusted,Persons in thousands,428,thousands,3,v2062809,1.1.1.1.1.1,29020.2,NaN,NaN,NaN,1
2,2015-06,Canada,2021A000011124,Population,Total - Gender,15 years and over,Estimate,Seasonally adjusted,Persons in thousands,428,thousands,3,v2062809,1.1.1.1.1.1,29048.8,NaN,NaN,NaN,1
3,2015-07,Canada,2021A000011124,Population,Total - Gender,15 years and over,Estimate,Seasonally adjusted,Persons in thousands,428,thousands,3,v2062809,1.1.1.1.1.1,29074.3,NaN,NaN,NaN,1
4,2015-08,Canada,2021A000011124,Population,Total - Gender,15 years and over,Estimate,Seasonally adjusted,Persons in thousands,428,thousands,3,v2062809,1.1.1.1.1.1,29099.5,NaN,NaN,NaN,1


In [7]:
# Remove the pivot column name
lfs_wide.columns.name = None

# Keep only required columns
lfs_clean = lfs_wide[[
    "month",
    "Employment",
    "Unemployment",
    "Labour force",
    "Population"
]].copy()

print("Clean LFS shape:", lfs_clean.shape)
display(lfs_clean.head())



Clean LFS shape: (117, 5)


,month,Employment,Unemployment,Labour force,Population
0,2015-04,17819.4,1337.2,19156.6,29000.4
1,2015-05,17843.7,1324.7,19168.4,29020.2
2,2015-06,17836.7,1326.6,19163.3,29048.8
3,2015-07,17863.3,1336.0,19199.2,29074.3
4,2015-08,17889.0,1358.3,19247.3,29099.5


In [8]:
# Convert from thousands to persons
lfs_clean["Employment"] = lfs_clean["Employment"] * 1000
lfs_clean["Unemployment"] = lfs_clean["Unemployment"] * 1000
lfs_clean["Labour force"] = lfs_clean["Labour force"] * 1000
lfs_clean["Population"] = lfs_clean["Population"] * 1000

print("Unit conversion complete.")
display(lfs_clean.head())


Unit conversion complete.


,month,Employment,Unemployment,Labour force,Population
0,2015-04,17819400.0,1337200.0,19156600.0,29000400.0
1,2015-05,17843700.0,1324700.0,19168400.0,29020200.0
2,2015-06,17836700.0,1326600.0,19163300.0,29048800.0
3,2015-07,17863300.0,1336000.0,19199200.0,29074300.0
4,2015-08,17889000.0,1358300.0,19247300.0,29099500.0


In [13]:
# Copy dataset
vac = vac_raw.copy()

# Filter: Canada + Total industry + Job vacancies
vac = vac[
    (vac["GEO"] == "Canada") &
    (vac["Statistics"] == "Job vacancies")
]

print("Filtered vacancy shape:", vac.shape)
vac.head()


Filtered vacancy shape: (111, 16)


,REF_DATE,GEO,DGUID,North American Industry Classification System (NAICS),Statistics,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,2015-04,Canada,2016A000011124,"Total, all industries",Job vacancies,Number,223,units,0,v1446283287,1.1.1,440425,A,NaN,NaN,0
1,2015-05,Canada,2016A000011124,"Total, all industries",Job vacancies,Number,223,units,0,v1446283287,1.1.1,407040,A,NaN,NaN,0
2,2015-06,Canada,2016A000011124,"Total, all industries",Job vacancies,Number,223,units,0,v1446283287,1.1.1,398910,A,NaN,NaN,0
3,2015-07,Canada,2016A000011124,"Total, all industries",Job vacancies,Number,223,units,0,v1446283287,1.1.1,376570,A,NaN,NaN,0
4,2015-08,Canada,2016A000011124,"Total, all industries",Job vacancies,Number,223,units,0,v1446283287,1.1.1,369190,A,NaN,NaN,0


In [16]:
# Keep only required columns
vac_clean = vac[["REF_DATE", "VALUE"]].copy()

# Rename columns
vac_clean = vac_clean.rename(columns={
    "REF_DATE": "month",
    "VALUE": "vacancies"
})

# Ensure proper datetime format
vac_clean["month"] = pd.to_datetime(vac_clean["month"])

# Sort chronologically
vac_clean = vac_clean.sort_values("month").reset_index(drop=True)

print("Clean Vacancy shape:", vac_clean.shape)
vac_clean.head()



Clean Vacancy shape: (111, 2)


,month,vacancies
0,2015-04-01,440425
1,2015-05-01,407040
2,2015-06-01,398910
3,2015-07-01,376570
4,2015-08-01,369190


In [17]:
# Ensure both are datetime
lfs_clean["month"] = pd.to_datetime(lfs_clean["month"])
vac_clean["month"] = pd.to_datetime(vac_clean["month"])

# Merge on month (left join keeps LFS full timeline)
df_merged = pd.merge(
    lfs_clean,
    vac_clean,
    on="month",
    how="left"
)

print("Merged dataset shape:", df_merged.shape)
df_merged.head()


Merged dataset shape: (117, 6)


,month,Employment,Unemployment,Labour force,Population,vacancies
0,2015-04-01,17819400.0,1337200.0,19156600.0,29000400.0,440425.0
1,2015-05-01,17843700.0,1324700.0,19168400.0,29020200.0,407040.0
2,2015-06-01,17836700.0,1326600.0,19163300.0,29048800.0,398910.0
3,2015-07-01,17863300.0,1336000.0,19199200.0,29074300.0,376570.0
4,2015-08-01,17889000.0,1358300.0,19247300.0,29099500.0,369190.0


In [18]:
# ==============================
# Missing Value Audit (Full)
# ==============================

missing_summary = pd.DataFrame({
    "Missing_Count": df_merged.isna().sum(),
    "Missing_Percentage": (df_merged.isna().sum() / len(df_merged)) * 100
})

missing_summary = missing_summary.sort_values(by="Missing_Count", ascending=False)

print("Total rows:", len(df_merged))
missing_summary


Total rows: 117


,Missing_Count,Missing_Percentage
vacancies,6,5.128205
month,0,0.000000
Employment,0,0.000000
Unemployment,0,0.000000
Labour force,0,0.000000
Population,0,0.000000


In [19]:
# Identify missing vacancy months
missing_vac = df_merged[df_merged["vacancies"].isna()]

print("Missing vacancy months:")
missing_vac[["month"]]


Missing vacancy months:


,month
60,2020-04-01
61,2020-05-01
62,2020-06-01
63,2020-07-01
64,2020-08-01
65,2020-09-01


In [20]:
# Remove months with missing vacancies
df_final = df_merged[df_merged["vacancies"].notna()].copy()

print("Final dataset shape:", df_final.shape)


Final dataset shape: (111, 6)


In [21]:
# ===============================
# Remove months with missing vacancies
# ===============================

df_final = df_merged[df_merged["vacancies"].notna()].copy()

print("Final dataset shape:", df_final.shape)

# Verify no missing values remain
print("\nMissing values after removal:")
print(df_final.isna().sum())
df_final.head()


Final dataset shape: (111, 6)

Missing values after removal:
month           0
Employment      0
Unemployment    0
Labour force    0
Population      0
vacancies       0
dtype: int64


,month,Employment,Unemployment,Labour force,Population,vacancies
0,2015-04-01,17819400.0,1337200.0,19156600.0,29000400.0,440425.0
1,2015-05-01,17843700.0,1324700.0,19168400.0,29020200.0,407040.0
2,2015-06-01,17836700.0,1326600.0,19163300.0,29048800.0,398910.0
3,2015-07-01,17863300.0,1336000.0,19199200.0,29074300.0,376570.0
4,2015-08-01,17889000.0,1358300.0,19247300.0,29099500.0,369190.0


In [22]:
# ===============================
# compute labour imbalance θ
# ===============================

df_final["theta"] = df_final["vacancies"] / df_final["Unemployment"]

print("\nTheta summary:")
print(df_final["theta"].describe())



Theta summary:
count    111.000000
mean       0.474081
std        0.186912
min        0.241134
25%        0.329083
50%        0.437226
75%        0.537656
max        0.976705
Name: theta, dtype: float64


In [23]:
# check missing values in theta
print("\nMissing values in theta:", df_final["theta"].isna().sum())
    


Missing values in theta: 0


## Outlier Detection – Indicator and Core Labour Variables

Before moving to predictive modeling, the dataset must be evaluated for extreme values that could distort time-series behavior and forecasting results.

Outliers are assessed using the Interquartile Range (IQR) method. This approach is robust to non-normal distributions and is appropriate for macroeconomic time-series data.

For each variable:

- Q1 = 25th percentile  
- Q3 = 75th percentile  
- IQR = Q3 − Q1  

Lower Bound = Q1 − 1.5 × IQR  
Upper Bound = Q3 + 1.5 × IQR  

Observations outside this range are flagged as potential outliers.

Variables evaluated:
- Job Vacancies  
- Unemployment  
- Employment  
- Labour Force  
- Labour Market Imbalance Indicator (θ = Vacancies / Unemployment)

The objective is not automatic removal, but structural validation.  
If outliers correspond to economic shocks (e.g., COVID-19), they are treated as structural events rather than data errors.


In [24]:
# =========================================
# Outlier Detection Function (IQR Method)
# =========================================

def detect_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = series[(series < lower_bound) | (series > upper_bound)]
    
    return lower_bound, upper_bound, outliers


In [25]:
# =========================================
# Apply IQR Outlier Detection
# =========================================

variables_to_check = [
    "vacancies",
    "Unemployment",
    "Employment",
    "Labour force",
    "theta"
]

outlier_summary = {}

for var in variables_to_check:
    lower, upper, outliers = detect_outliers_iqr(df_final[var])
    
    outlier_summary[var] = {
        "Lower_Bound": lower,
        "Upper_Bound": upper,
        "Outlier_Count": len(outliers)
    }

# Convert to DataFrame for clean display
import pandas as pd
outlier_results = pd.DataFrame(outlier_summary).T

outlier_results


,Lower_Bound,Upper_Bound,Outlier_Count
vacancies,8.438250e+04,1.034082e+06,0.0
Unemployment,8.321250e+05,1.685125e+06,5.0
Employment,1.626668e+07,2.190688e+07,0.0
Labour force,1.784840e+07,2.267320e+07,0.0
theta,1.622485e-02,8.505145e-01,7.0


### Outlier Interpretation

Outlier detection identified:

- 5 outliers in Unemployment
- 7 outliers in θ (Vacancy-to-Unemployment ratio)

No outliers were detected in vacancies, employment, or labour force.

The identified extreme values correspond to the COVID-19 economic shock period, where unemployment rose sharply while vacancies declined. These values represent structural macroeconomic disruption rather than measurement error.

Therefore, no outliers are removed from the dataset. These observations are retained as they reflect genuine labour market imbalance dynamics and are critical for forecasting and scenario analysis.


## Time-Series Structural Validation

Before proceeding to forecasting, the dataset must be validated as a proper monthly time series.

The following checks are performed:

1. Chronological sorting
2. Duplicate month verification
3. Monthly continuity validation
4. Structural gap confirmation (COVID suspension period)

This ensures the dataset is stable and suitable for time-series modeling.


In [26]:
# =========================================
# Time-Series Structural Validation
# =========================================

# Ensure datetime format
df_final["month"] = pd.to_datetime(df_final["month"])

# Sort chronologically
df_final = df_final.sort_values("month").reset_index(drop=True)

# Check duplicates
duplicate_months = df_final["month"].duplicated().sum()

# Generate full monthly range
full_range = pd.date_range(
    start=df_final["month"].min(),
    end=df_final["month"].max(),
    freq="MS"
)

missing_months = set(full_range) - set(df_final["month"])

print("Duplicate months:", duplicate_months)
print("Missing months count:", len(missing_months))
print("Missing months:", sorted(missing_months))


Duplicate months: 0
Missing months count: 6
Missing months: [Timestamp('2020-04-01 00:00:00'), Timestamp('2020-05-01 00:00:00'), Timestamp('2020-06-01 00:00:00'), Timestamp('2020-07-01 00:00:00'), Timestamp('2020-08-01 00:00:00'), Timestamp('2020-09-01 00:00:00')]


## Final Modeling Dataset Construction

The final dataset excludes the COVID-19 vacancy suspension period (April–September 2020) to avoid structural reporting bias.

The dataset is:

- Monthly
- Chronologically ordered
- Free from duplicate timestamps
- Free from missing values
- Structurally validated

This dataset will be used for:

- Descriptive analysis
- Diagnostic analysis
- Predictive forecasting
- Scenario simulation


In [27]:
# =========================================
# Create Final Modeling Dataset
# =========================================

model_df = df_final[[
    "month",
    "Employment",
    "Unemployment",
    "Labour force",
    "Population",
    "vacancies",
    "theta"
]].copy()

print("Model dataset shape:", model_df.shape)
model_df.head()


Model dataset shape: (111, 7)


,month,Employment,Unemployment,Labour force,Population,vacancies,theta
0,2015-04-01,17819400.0,1337200.0,19156600.0,29000400.0,440425.0,0.329364
1,2015-05-01,17843700.0,1324700.0,19168400.0,29020200.0,407040.0,0.307270
2,2015-06-01,17836700.0,1326600.0,19163300.0,29048800.0,398910.0,0.300701
3,2015-07-01,17863300.0,1336000.0,19199200.0,29074300.0,376570.0,0.281864
4,2015-08-01,17889000.0,1358300.0,19247300.0,29099500.0,369190.0,0.271803


In [28]:
# Save clean modeling dataset
model_df.to_csv("../1_data/processed/final_dataset.csv", index=False)

print("Final dataset exported successfully.")


Final dataset exported successfully.
